# Demo sin WhatsApp (plan B)

Invoca el orquestador real (Gemini + Postgres, checkpointer real) directamente en Python, sin pasar por el canal de WhatsApp -- para la defensa, si el túnel HTTPS o la configuración de Meta fallan el día de la demo (ver `docs/setup-whatsapp.md` y plandefases.md Fase 8, tarea 10).

Requiere: `docker compose up -d db` corriendo y al menos una `GEMINI_API_KEY_*` real en `.env` (no hace falta `USE_FIXTURES=false`, esta notebook no lee ese flag -- arma el agente real explícitamente).

In [ ]:
import os
import sys
import uuid
from pathlib import Path

# Jupyter fija el cwd en la carpeta de la notebook; nos paramos en la raíz
# del proyecto para que config.py encuentre .env con una ruta relativa.
os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd() / "src"))

import psycopg

from fitosanitarios.config import get_settings
from fitosanitarios.orquestador.agente import (
    checkpointer_postgres,
    crear_agente,
    crear_modelo_chat_gemini,
    modelos_de_respaldo,
)
from fitosanitarios.orquestador.estado import ContadorRepreguntas
from fitosanitarios.orquestador.turno import ejecutar_turno

settings = get_settings()
assert settings.gemini_api_keys, "Falta al menos una GEMINI_API_KEY_* en .env para esta demo"
print(f"Modelo: {settings.gemini_model} -- {len(settings.gemini_api_keys)} key(s) configurada(s)")

# Id único por corrida: el checkpointer es Postgres real y persiste entre
# corridas de esta notebook. Con un thread_id fijo, repetir un ensayo antes
# de la defensa hace que el LLM "recuerde" la conversación anterior y a
# veces no vuelva a llamar la tool (ver DIFICULTADES.md, Fase 10) -- por
# eso cada corrida arranca threads nuevos.
SESION = uuid.uuid4().hex[:8]
print(f"Sesión de esta corrida: {SESION}")

In [ ]:
model = crear_modelo_chat_gemini(settings)
pila_checkpointer = checkpointer_postgres(settings.database_url)
checkpointer = pila_checkpointer.__enter__()
# Las otras keys, para rotar si una se queda sin cuota.
agente = crear_agente(
    model, checkpointer=checkpointer, respaldo=modelos_de_respaldo(settings)
)
contador = ContadorRepreguntas()
conn_log = psycopg.connect(settings.database_url)
print("Agente listo.")

In [ ]:
def turno(thread_id: str, texto: str) -> None:
    print(f"\n\U0001f9d1\u200d\U0001f33e ({thread_id}): {texto}")
    _, mensajes = ejecutar_turno(agente, thread_id, texto, contador, conn_log=conn_log)
    for m in mensajes:
        print(f"\U0001f916: {m}\n")

## Caso 1 -- saludo / ayuda (fuera de una tool puntual)

In [ ]:
turno(f"demo-notebook-{SESION}-1", "Hola, ¿qué podés hacer?")

## Caso 2 -- validar un producto puntual contra SENASA

In [ ]:
turno(f"demo-notebook-{SESION}-2", "¿El Flyer 10 Ec está habilitado para soja?")

## Caso 3 -- listado de productos por adversidad y cultivo

In [ ]:
turno(f"demo-notebook-{SESION}-3", "¿Qué productos hay registrados para yuyo colorado en soja?")

## Caso 4 -- dictamen completo (producto + dosis + localidad)

Desde el 19/09/2026 el dictamen identifica la localidad por nombre (ver `DECISIONES.md`): El Trébol, con normativa real cargada.

In [ ]:
turno(
    f"demo-notebook-{SESION}-4",
    "Quiero validar la receta completa: Flyer 10 Ec en soja, 170 cm3/ha, terrestre, "
    "en El Trébol, contra chinche de la alfalfa",
)

## Caso 5 -- consulta de normativa con repregunta (falta la localidad) y follow-up

In [ ]:
turno(f"demo-notebook-{SESION}-5", "¿A cuántos metros de una escuela puedo aplicar por tierra?")
turno(f"demo-notebook-{SESION}-5", "Sastre")

## Caso 6 -- fuera de dominio

In [ ]:
turno(f"demo-notebook-{SESION}-6", "¿Va a llover mañana en Rosario?")

## Caso 7 -- Fase 9: registrar una aplicación real y consultar la agenda

In [ ]:
turno(f"demo-notebook-{SESION}-7", "Empecé a aplicar con la mosquito en el lote 4")
turno(f"demo-notebook-{SESION}-7", "Terminé de aplicar")
turno(f"demo-notebook-{SESION}-7", "¿qué tengo para hoy?")

## Limpieza

In [ ]:
conn_log.close()
pila_checkpointer.__exit__(None, None, None)
print("Conexiones cerradas.")